In [ ]:
import json
import csv

def json_to_csv(json_file_path, csv_file_path):
    """Convert JSON data to CSV format"""
    try:
        # Read JSON data from file
        with open(json_file_path, 'r', encoding='utf-8') as json_file:
            data = json.load(json_file)
        
        # Check if data is a list
        if not isinstance(data, list):
            print("Error: JSON data should be a list of objects")
            return
        
        # Get field names from the first object
        if data:
            fieldnames = data[0].keys()
        else:
            print("Error: Empty JSON data")
            return
        
        # Write to CSV
        with open(csv_file_path, 'w', newline='', encoding='utf-8') as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data)
        
        print(f"Successfully converted {json_file_path} to {csv_file_path}")
        print(f"Total records: {len(data)}")
        
    except FileNotFoundError:
        print(f"Error: File {json_file_path} not found")
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON format in {json_file_path}")
    except Exception as e:
        print(f"Error: {str(e)}")

# Convert urls.json to CSV
if __name__ == "__main__":
    json_to_csv('urls.json', 'publications.csv')

Successfully converted urls.json to publications.csv
Total records: 71


In [6]:
import pandas as pd

def join_csv_files(publications_file, rankings_file, output_file):
    """Join two CSV files based on publication_external_id using outer join to preserve all data"""
    try:
        # Read both CSV files
        print("Reading CSV files...")
        publications_df = pd.read_csv(publications_file)
        rankings_df = pd.read_csv(rankings_file)
        
        print(f"Publications CSV: {len(publications_df)} records")
        print(f"Rankings CSV: {len(rankings_df)} records")
        
        # Display column names to verify the join key
        print(f"\nPublications columns: {list(publications_df.columns)}")
        print(f"Rankings columns: {list(rankings_df.columns)}")
        
        # Perform OUTER join to keep all data from both files
        merged_df = pd.merge(
            publications_df, 
            rankings_df, 
            on='publication_external_id', 
            how='outer',
            indicator=True  # This adds a column showing the source of each row
        )
        
        print(f"\nAfter outer join: {len(merged_df)} records")
        
        # Analyze mismatches and missing data
        print("\n=== MISMATCH ANALYSIS ===")
        
        # Count records by source
        merge_counts = merged_df['_merge'].value_counts()
        print(f"Records in both files: {merge_counts.get('both', 0)}")
        print(f"Records only in publications: {merge_counts.get('left_only', 0)}")
        print(f"Records only in rankings: {merge_counts.get('right_only', 0)}")
        
        # Show publications without rankings
        publications_without_rankings = merged_df[merged_df['_merge'] == 'left_only']
        if len(publications_without_rankings) > 0:
            print(f"\n⚠️  PUBLICATIONS WITHOUT RANKINGS ({len(publications_without_rankings)} records):")
            for idx, row in publications_without_rankings.iterrows():
                print(f"  - ID: {row['publication_external_id']} | Title: {row['publication_title'][:60]}...")
        
        # Show rankings without publications
        rankings_without_publications = merged_df[merged_df['_merge'] == 'right_only']
        if len(rankings_without_publications) > 0:
            print(f"\n⚠️  RANKINGS WITHOUT PUBLICATIONS ({len(rankings_without_publications)} records):")
            for idx, row in rankings_without_publications.iterrows():
                print(f"  - ID: {row['publication_external_id']}")
        
        # Check for null URLs in publications
        null_urls = merged_df[merged_df['url'].isnull() & (merged_df['_merge'] != 'right_only')]
        if len(null_urls) > 0:
            print(f"\n⚠️  PUBLICATIONS WITH NULL URLs ({len(null_urls)} records):")
            for idx, row in null_urls.iterrows():
                print(f"  - ID: {row['publication_external_id']} | Title: {row['publication_title'][:60]}...")
        
        # Save the merged data (keeping the _merge column for reference)
        merged_df.to_csv(output_file, index=False)
        print(f"\nSuccessfully saved joined data to {output_file}")
        
        # Create a clean version without the _merge column
        clean_df = merged_df.drop('_merge', axis=1)
        clean_output = output_file.replace('.csv', '_clean.csv')
        clean_df.to_csv(clean_output, index=False)
        print(f"Clean version (without _merge column) saved to {clean_output}")
        
        # Display first few rows to verify the join
        print(f"\nFirst 5 rows of joined data:")
        print(merged_df[['publication_external_id', 'publication_title', '_merge']].head())
        
        # Show comprehensive summary
        print(f"\n=== COMPREHENSIVE SUMMARY ===")
        print(f"- Total publications: {len(publications_df)}")
        print(f"- Total rankings: {len(rankings_df)}")
        print(f"- Successfully matched: {merge_counts.get('both', 0)}")
        print(f"- Publications without rankings: {merge_counts.get('left_only', 0)}")
        print(f"- Rankings without publications: {merge_counts.get('right_only', 0)}")
        print(f"- Total records in output: {len(merged_df)}")
        print(f"- Data preservation: 100% (no data lost)")
        
        return merged_df
        
    except FileNotFoundError as e:
        print(f"Error: File not found - {e}")
    except KeyError as e:
        print(f"Error: Column not found - {e}")
        print("Make sure both files have 'publication_external_id' column")
    except Exception as e:
        print(f"Error: {str(e)}")

def analyze_data_quality(merged_df):
    """Additional data quality analysis"""
    print(f"\n=== DATA QUALITY ANALYSIS ===")
    
    # Check for duplicate publication_external_ids
    duplicates = merged_df['publication_external_id'].duplicated()
    if duplicates.any():
        print(f"⚠️  Found {duplicates.sum()} duplicate publication IDs")
        duplicate_ids = merged_df[duplicates]['publication_external_id'].tolist()
        print(f"Duplicate IDs: {duplicate_ids}")
    else:
        print("✅ No duplicate publication IDs found")
    
    # Check for missing publication titles
    missing_titles = merged_df['publication_title'].isnull().sum()
    if missing_titles > 0:
        print(f"⚠️  Found {missing_titles} records with missing publication titles")
    else:
        print("✅ All records have publication titles")

# Main execution
if __name__ == "__main__":
    # Perform outer join to preserve all data
    joined_df = join_csv_files(
        'publications.csv', 
        'combined_ranking_sorted.csv', 
        'pub_merged.csv'
    )
    
    # Perform additional data quality analysis
    if joined_df is not None:
        analyze_data_quality(joined_df)

Reading CSV files...
Publications CSV: 71 records
Rankings CSV: 71 records

Publications columns: ['publication_external_id', 'publication_title', 'url']
Rankings columns: ['publication_external_id', 'rag_score', 'rag_total', 'rag_percentage', 'general_score', 'general_total', 'general_percentage', 'combined_percentage']

After outer join: 71 records

=== MISMATCH ANALYSIS ===
Records in both files: 71
Records only in publications: 0
Records only in rankings: 0

⚠️  PUBLICATIONS WITH NULL URLs (3 records):
  - ID: 52Zo0G7WuWGZ | Title: AAIDC2025- Project 1- RAG AI Assistant ...
  - ID: HWb7EKooLzJS | Title: OptiBotSync – Ideal for productivity or multi-tool integrati...
  - ID: SL2d7v43ml6X | Title: testing test...

Successfully saved joined data to pub_merged.csv
Clean version (without _merge column) saved to pub_merged_clean.csv

First 5 rows of joined data:
  publication_external_id                                  publication_title  \
0            00LAGp5OuigT                      

In [8]:
import pandas as pd

def merge_results_files(rag_file, general_file, output_file):
    """Merge rag_results.csv and general_results.csv based on project column"""
    try:
        # Read both CSV files
        print("Reading results CSV files...")
        rag_df = pd.read_csv(rag_file)
        general_df = pd.read_csv(general_file)
        
        print(f"RAG results CSV: {len(rag_df)} records")
        print(f"General results CSV: {len(general_df)} records")
        
        # Display column names to verify the join key
        print(f"\nRAG results columns: {list(rag_df.columns)}")
        print(f"General results columns: {list(general_df.columns)}")
        
        # Check if 'project' column exists in both files
        if 'project' not in rag_df.columns:
            print("Error: 'project' column not found in rag_results.csv")
            return None
        if 'project' not in general_df.columns:
            print("Error: 'project' column not found in general_results.csv")
            return None
        
        # Add suffixes to distinguish columns from different files
        merged_df = pd.merge(
            rag_df, 
            general_df, 
            on='project', 
            how='outer',
            suffixes=('_rag', '_general'),
            indicator=True
        )
        
        print(f"\nAfter outer merge: {len(merged_df)} records")
        
        # Analyze merge results
        print("\n=== MERGE ANALYSIS ===")
        
        merge_counts = merged_df['_merge'].value_counts()
        print(f"Projects in both files: {merge_counts.get('both', 0)}")
        print(f"Projects only in RAG results: {merge_counts.get('left_only', 0)}")
        print(f"Projects only in General results: {merge_counts.get('right_only', 0)}")
        
        # Show projects only in RAG results
        rag_only = merged_df[merged_df['_merge'] == 'left_only']
        if len(rag_only) > 0:
            print(f"\n⚠️  PROJECTS ONLY IN RAG RESULTS ({len(rag_only)} records):")
            for idx, row in rag_only.iterrows():
                print(f"  - Project: {row['project']}")
        
        # Show projects only in General results
        general_only = merged_df[merged_df['_merge'] == 'right_only']
        if len(general_only) > 0:
            print(f"\n⚠️  PROJECTS ONLY IN GENERAL RESULTS ({len(general_only)} records):")
            for idx, row in general_only.iterrows():
                print(f"  - Project: {row['project']}")
        
        # Save the merged data
        merged_df.to_csv(output_file, index=False)
        print(f"\nSuccessfully saved merged data to {output_file}")
        
        # Create a clean version without the _merge column
        clean_df = merged_df.drop('_merge', axis=1)
        clean_output = output_file.replace('.csv', '_clean.csv')
        clean_df.to_csv(clean_output, index=False)
        print(f"Clean version saved to {clean_output}")
        
        # Display first few rows
        print(f"\nFirst 5 rows of merged data:")
        display_cols = ['project', '_merge'] + [col for col in merged_df.columns if col not in ['project', '_merge']][:5]
        print(merged_df[display_cols].head())
        
        # Show comprehensive summary
        print(f"\n=== COMPREHENSIVE SUMMARY ===")
        print(f"- Total RAG results: {len(rag_df)}")
        print(f"- Total General results: {len(general_df)}")
        print(f"- Successfully matched projects: {merge_counts.get('both', 0)}")
        print(f"- RAG-only projects: {merge_counts.get('left_only', 0)}")
        print(f"- General-only projects: {merge_counts.get('right_only', 0)}")
        print(f"- Total records in output: {len(merged_df)}")
        print(f"- Data preservation: 100% (no data lost)")
        
        return merged_df
        
    except FileNotFoundError as e:
        print(f"Error: File not found - {e}")
    except KeyError as e:
        print(f"Error: Column not found - {e}")
    except Exception as e:
        print(f"Error: {str(e)}")

def analyze_merged_data(merged_df):
    """Analyze the merged data for insights"""
    print(f"\n=== MERGED DATA ANALYSIS ===")
    
    # Check for duplicate projects
    duplicates = merged_df['project'].duplicated()
    if duplicates.any():
        print(f"⚠️  Found {duplicates.sum()} duplicate projects")
        duplicate_projects = merged_df[duplicates]['project'].tolist()
        print(f"Duplicate projects: {duplicate_projects}")
    else:
        print("✅ No duplicate projects found")
    
    # Check for missing values in key columns
    print(f"\nMissing values analysis:")
    for col in merged_df.columns:
        if col != '_merge':
            missing_count = merged_df[col].isnull().sum()
            if missing_count > 0:
                print(f"  - {col}: {missing_count} missing values")
    
    # Show column overlap
    rag_cols = set([col.replace('_rag', '') for col in merged_df.columns if col.endswith('_rag')])
    general_cols = set([col.replace('_general', '') for col in merged_df.columns if col.endswith('_general')])
    common_metrics = rag_cols.intersection(general_cols)
    
    if common_metrics:
        print(f"\n📊 Common metrics between RAG and General results:")
        for metric in common_metrics:
            print(f"  - {metric}")

def create_comparison_report(merged_df):
    """Create a comparison report for projects that exist in both files"""
    both_projects = merged_df[merged_df['_merge'] == 'both'].copy()
    
    if len(both_projects) == 0:
        print("No projects found in both files for comparison")
        return
    
    print(f"\n=== COMPARISON REPORT FOR {len(both_projects)} COMMON PROJECTS ===")
    
    # Find numeric columns for comparison
    numeric_cols = both_projects.select_dtypes(include=['number']).columns
    rag_numeric = [col for col in numeric_cols if col.endswith('_rag')]
    general_numeric = [col for col in numeric_cols if col.endswith('_general')]
    
    print(f"Numeric columns available for comparison:")
    print(f"  - RAG columns: {len(rag_numeric)}")
    print(f"  - General columns: {len(general_numeric)}")
    
    # Save comparison data
    comparison_file = 'results_comparison.csv'
    both_projects.to_csv(comparison_file, index=False)
    print(f"Comparison data saved to {comparison_file}")

# Main execution
if __name__ == "__main__":
    # Merge the results files
    merged_df = merge_results_files(
        'rag_results.csv',
        'general_results.csv', 
        'repo_merged_results.csv'
    )
    
    # Perform additional analysis
    if merged_df is not None:
        analyze_merged_data(merged_df)
        create_comparison_report(merged_df)

Reading results CSV files...
RAG results CSV: 65 records
General results CSV: 66 records

RAG results columns: ['project', 'total_criteria', 'met_criteria', 'percentage', 'timestamp', 'essential_met', 'essential_total', 'essential_percentage', 'professional_met', 'professional_total', 'professional_percentage', 'elite_met', 'elite_total', 'elite_percentage']
General results columns: ['project', 'total_criteria', 'met_criteria', 'percentage', 'timestamp', 'essential_met', 'essential_total', 'essential_percentage', 'professional_met', 'professional_total', 'professional_percentage', 'elite_met', 'elite_total', 'elite_percentage']

After outer merge: 66 records

=== MERGE ANALYSIS ===
Projects in both files: 66
Projects only in RAG results: 0
Projects only in General results: 0

Successfully saved merged data to repo_merged_results.csv
Clean version saved to repo_merged_results_clean.csv

First 5 rows of merged data:
                                             project _merge  \
0  https:

In [12]:
import pandas as pd

def merge_repo_pub_files(repo_file, pub_file, output_file):
    """Merge repo_merged_results.csv and pub_merged.csv based on project column"""
    try:
        # Read both CSV files
        print("Reading repo and publication CSV files...")
        repo_df = pd.read_csv(repo_file)
        pub_df = pd.read_csv(pub_file)
        
        print(f"Repo merged results CSV: {len(repo_df)} records")
        print(f"Publication merged CSV: {len(pub_df)} records")
        
        # Display column names to verify the join key
        print(f"\nRepo results columns: {list(repo_df.columns)}")
        print(f"Publication columns: {list(pub_df.columns)}")
        
        # Check if 'project' column exists in both files
        if 'project' not in repo_df.columns:
            print("Error: 'project' column not found in repo_merged_results.csv")
            return None
        if 'project' not in pub_df.columns:
            print("Error: 'project' column not found in pub_merged.csv")
            return None
        
        # Add suffixes to distinguish columns from different files
        merged_df = pd.merge(
            repo_df, 
            pub_df, 
            on='project', 
            how='outer',
            suffixes=('_repo', '_pub'),
            indicator=True
        )
        
        print(f"\nAfter outer merge: {len(merged_df)} records")
        
        # Analyze merge results
        print("\n=== MERGE ANALYSIS ===")
        
        merge_counts = merged_df['_merge'].value_counts()
        print(f"Projects in both files: {merge_counts.get('both', 0)}")
        print(f"Projects only in Repo results: {merge_counts.get('left_only', 0)}")
        print(f"Projects only in Publications: {merge_counts.get('right_only', 0)}")
        
        # Show projects only in Repo results
        repo_only = merged_df[merged_df['_merge'] == 'left_only']
        if len(repo_only) > 0:
            print(f"\n⚠️  PROJECTS ONLY IN REPO RESULTS ({len(repo_only)} records):")
            for idx, row in repo_only.iterrows():
                print(f"  - Project: {row['project']}")
        
        # Show projects only in Publications
        pub_only = merged_df[merged_df['_merge'] == 'right_only']
        if len(pub_only) > 0:
            print(f"\n⚠️  PROJECTS ONLY IN PUBLICATIONS ({len(pub_only)} records):")
            for idx, row in pub_only.iterrows():
                print(f"  - Project: {row['project']}")
        
        # Save the merged data
        merged_df.to_csv(output_file, index=False)
        print(f"\nSuccessfully saved merged data to {output_file}")
        
        # Create a clean version without the _merge column
        clean_df = merged_df.drop('_merge', axis=1)
        clean_output = output_file.replace('.csv', '_clean.csv')
        clean_df.to_csv(clean_output, index=False)
        print(f"Clean version saved to {clean_output}")
        
        # Display first few rows
        print(f"\nFirst 5 rows of merged data:")
        display_cols = ['project', '_merge'] + [col for col in merged_df.columns if col not in ['project', '_merge']][:5]
        if len(display_cols) > 7:
            display_cols = display_cols[:7]  # Limit display for readability
        print(merged_df[display_cols].head())
        
        # Show comprehensive summary
        print(f"\n=== COMPREHENSIVE SUMMARY ===")
        print(f"- Total Repo results: {len(repo_df)}")
        print(f"- Total Publications: {len(pub_df)}")
        print(f"- Successfully matched projects: {merge_counts.get('both', 0)}")
        print(f"- Repo-only projects: {merge_counts.get('left_only', 0)}")
        print(f"- Publication-only projects: {merge_counts.get('right_only', 0)}")
        print(f"- Total records in output: {len(merged_df)}")
        print(f"- Data preservation: 100% (no data lost)")
        
        return merged_df
        
    except FileNotFoundError as e:
        print(f"Error: File not found - {e}")
    except KeyError as e:
        print(f"Error: Column not found - {e}")
    except Exception as e:
        print(f"Error: {str(e)}")

def analyze_repo_pub_data(merged_df):
    """Analyze the merged repo and publication data"""
    print(f"\n=== REPO & PUBLICATION DATA ANALYSIS ===")
    
    # Check for duplicate projects
    duplicates = merged_df['project'].duplicated()
    if duplicates.any():
        print(f"⚠️  Found {duplicates.sum()} duplicate projects")
        duplicate_projects = merged_df[duplicates]['project'].tolist()
        print(f"Duplicate projects: {duplicate_projects}")
    else:
        print("✅ No duplicate projects found")
    
    # Check for missing values in key columns
    print(f"\nMissing values analysis:")
    key_columns = ['project']
    
    # Add common publication columns if they exist
    pub_cols = [col for col in merged_df.columns if col.endswith('_pub')]
    repo_cols = [col for col in merged_df.columns if col.endswith('_repo')]
    
    print(f"  - Repository columns: {len(repo_cols)}")
    print(f"  - Publication columns: {len(pub_cols)}")
    
    for col in merged_df.columns:
        if col not in ['_merge'] and merged_df[col].isnull().sum() > 0:
            missing_count = merged_df[col].isnull().sum()
            print(f"  - {col}: {missing_count} missing values")
    
    # Analyze projects with both repo and publication data
    both_data = merged_df[merged_df['_merge'] == 'both']
    print(f"\n📊 Projects with both repo and publication data: {len(both_data)}")
    
    if len(both_data) > 0:
        # Check for URL matching if URLs exist in both datasets
        url_cols_repo = [col for col in both_data.columns if 'url' in col.lower() and col.endswith('_repo')]
        url_cols_pub = [col for col in both_data.columns if 'url' in col.lower() and col.endswith('_pub')]
        
        if url_cols_repo and url_cols_pub:
            print(f"Found URL columns for comparison:")
            print(f"  - Repo URLs: {url_cols_repo}")
            print(f"  - Publication URLs: {url_cols_pub}")

def create_project_summary(merged_df):
    """Create a project summary report"""
    print(f"\n=== PROJECT SUMMARY REPORT ===")
    
    # Count projects by data availability
    both_data = len(merged_df[merged_df['_merge'] == 'both'])
    repo_only = len(merged_df[merged_df['_merge'] == 'left_only'])
    pub_only = len(merged_df[merged_df['_merge'] == 'right_only'])
    
    print(f"Data availability breakdown:")
    print(f"  - Complete data (repo + publication): {both_data}")
    print(f"  - Repository data only: {repo_only}")
    print(f"  - Publication data only: {pub_only}")
    print(f"  - Total unique projects: {both_data + repo_only + pub_only}")
    
    # Save summary report
    summary_data = {
        'Category': ['Complete Data (Repo + Pub)', 'Repository Only', 'Publication Only', 'Total Projects'],
        'Count': [both_data, repo_only, pub_only, both_data + repo_only + pub_only],
        'Percentage': [
            f"{both_data/len(merged_df)*100:.1f}%",
            f"{repo_only/len(merged_df)*100:.1f}%", 
            f"{pub_only/len(merged_df)*100:.1f}%",
            "100.0%"
        ]
    }
    
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv('project_summary_report.csv', index=False)
    print(f"\nSummary report saved to project_summary_report.csv")
    print(summary_df.to_string(index=False))

# Main execution
if __name__ == "__main__":
    # Merge the repo and publication files
    merged_df = merge_repo_pub_files(
        'repo_merged_results.csv',
        'pub_merged.csv', 
        'final_merged_data.csv'
    )
    
    # Perform additional analysis
    if merged_df is not None:
        analyze_repo_pub_data(merged_df)
        create_project_summary(merged_df)

Reading repo and publication CSV files...
Repo merged results CSV: 66 records
Publication merged CSV: 70 records

Repo results columns: ['project', 'total_criteria_rag', 'met_criteria_rag', 'percentage_rag', 'timestamp_rag', 'essential_met_rag', 'essential_total_rag', 'essential_percentage_rag', 'professional_met_rag', 'professional_total_rag', 'professional_percentage_rag', 'elite_met_rag', 'elite_total_rag', 'elite_percentage_rag', 'total_criteria_general', 'met_criteria_general', 'percentage_general', 'timestamp_general', 'essential_met_general', 'essential_total_general', 'essential_percentage_general', 'professional_met_general', 'professional_total_general', 'professional_percentage_general', 'elite_met_general', 'elite_total_general', 'elite_percentage_general', 'merge1']
Publication columns: ['publication_external_id', 'publication_title', 'project', 'rag_score', 'rag_total', 'rag_percentage', 'general_score', 'general_total', 'general_percentage', 'combined_percentage', 'merge

In [18]:
import pandas as pd

def check_publication_duplicates(file_path):
    """Check for duplicates based on publication_external_id in final_merged_data.csv"""
    try:
        # Read the merged data
        print(f"Reading {file_path}...")
        df = pd.read_csv(file_path)
        
        print(f"Total records: {len(df)}")
        print(f"Columns: {list(df.columns)}")
        
        # Check if publication_external_id column exists
        pub_id_cols = [col for col in df.columns if 'publication_external_id' in col.lower()]
        
        if not pub_id_cols:
            print("❌ No publication_external_id column found")
            print("Available columns that might contain publication ID:")
            potential_cols = [col for col in df.columns if any(keyword in col.lower() for keyword in ['publication', 'pub', 'external', 'id'])]
            for col in potential_cols:
                print(f"  - {col}")
            return
        
        print(f"\nFound publication ID columns: {pub_id_cols}")
        
        # Check each publication_external_id column for duplicates
        for col in pub_id_cols:
            print(f"\n=== ANALYZING {col} ===")
            
            # Remove null values for duplicate analysis
            non_null_values = df[col].dropna()
            print(f"Non-null values: {len(non_null_values)} out of {len(df)}")
            
            if len(non_null_values) == 0:
                print("⚠️  All values are null - cannot check for duplicates")
                continue
            
            # Check for duplicates
            duplicates = non_null_values.duplicated()
            duplicate_count = duplicates.sum()
            
            if duplicate_count > 0:
                print(f"🚨 FOUND {duplicate_count} DUPLICATE VALUES")
                
                # Get the duplicate values
                duplicate_values = non_null_values[duplicates].unique()
                print(f"Duplicate publication IDs: {list(duplicate_values)}")
                
                # Show detailed information about duplicates
                for dup_id in duplicate_values:
                    dup_rows = df[df[col] == dup_id]
                    print(f"\n  📋 Publication ID '{dup_id}' appears {len(dup_rows)} times:")
                    
                    for idx, row in dup_rows.iterrows():
                        project = row.get('project', 'N/A')
                        title_col = next((c for c in df.columns if 'title' in c.lower()), None)
                        title = row.get(title_col, 'N/A') if title_col else 'N/A'
                        print(f"    - Row {idx}: Project='{project}', Title='{str(title)[:50]}...'")
                
                # Create a report of duplicate rows
                duplicate_rows = df[df[col].isin(duplicate_values) & df[col].notna()]
                duplicate_report_file = f'duplicate_publications_{col}.csv'
                duplicate_rows.to_csv(duplicate_report_file, index=False)
                print(f"\n💾 Duplicate rows saved to: {duplicate_report_file}")
                
            else:
                print("✅ No duplicates found")
            
            # Show some statistics
            unique_count = non_null_values.nunique()
            print(f"📊 Statistics:")
            print(f"  - Total non-null values: {len(non_null_values)}")
            print(f"  - Unique values: {unique_count}")
            print(f"  - Duplicates: {duplicate_count}")
            print(f"  - Duplicate rate: {(duplicate_count/len(non_null_values)*100):.2f}%")
    
    except FileNotFoundError:
        print(f"❌ Error: File '{file_path}' not found")
    except Exception as e:
        print(f"❌ Error reading file: {e}")

def analyze_data_quality(file_path):
    """Additional data quality analysis"""
    try:
        df = pd.read_csv(file_path)
        
        print(f"\n=== OVERALL DATA QUALITY ANALYSIS ===")
        
        # Check for completely duplicate rows
        complete_duplicates = df.duplicated()
        if complete_duplicates.any():
            print(f"🚨 Found {complete_duplicates.sum()} completely duplicate rows")
            # Save complete duplicates
            duplicate_rows = df[complete_duplicates]
            duplicate_rows.to_csv('complete_duplicate_rows.csv', index=False)
            print("Complete duplicate rows saved to: complete_duplicate_rows.csv")
        else:
            print("✅ No completely duplicate rows found")
        
        # Check project duplicates
        if 'project' in df.columns:
            project_duplicates = df['project'].duplicated()
            if project_duplicates.any():
                print(f"🚨 Found {project_duplicates.sum()} duplicate project names")
                duplicate_projects = df[project_duplicates]['project'].unique()
                print(f"Duplicate projects: {list(duplicate_projects)}")
            else:
                print("✅ No duplicate project names found")
        
        # Summary of null values
        print(f"\n📊 Missing Data Summary:")
        null_counts = df.isnull().sum()
        for col in null_counts[null_counts > 0].index:
            print(f"  - {col}: {null_counts[col]} null values ({null_counts[col]/len(df)*100:.1f}%)")
            
    except Exception as e:
        print(f"Error in data quality analysis: {e}")

def create_deduplication_report(file_path):
    """Create a comprehensive deduplication report"""
    try:
        df = pd.read_csv(file_path)
        
        # Find all potential ID columns
        id_columns = [col for col in df.columns if any(keyword in col.lower() for keyword in ['id', 'external'])]
        
        report_data = []
        
        for col in id_columns:
            non_null = df[col].dropna()
            if len(non_null) > 0:
                unique_count = non_null.nunique()
                duplicate_count = len(non_null) - unique_count
                
                report_data.append({
                    'Column': col,
                    'Total_Values': len(non_null),
                    'Unique_Values': unique_count,
                    'Duplicates': duplicate_count,
                    'Duplicate_Rate_%': round((duplicate_count/len(non_null)*100), 2) if len(non_null) > 0 else 0
                })
        
        if report_data:
            report_df = pd.DataFrame(report_data)
            report_df.to_csv('deduplication_report.csv', index=False)
            print(f"\n📄 Deduplication report saved to: deduplication_report.csv")
            print(report_df.to_string(index=False))
        
    except Exception as e:
        print(f"Error creating deduplication report: {e}")

# Main execution
if __name__ == "__main__":
    file_path = 'final_merged_data.csv'
    
    # Check for publication_external_id duplicates
    check_publication_duplicates(file_path)
    
    # Additional data quality analysis
    analyze_data_quality(file_path)
    
    # Create comprehensive report
    create_deduplication_report(file_path)

Reading final_merged_data.csv...
Total records: 70
Columns: ['project', 'total_criteria_rag', 'met_criteria_rag', 'percentage_rag', 'timestamp_rag', 'essential_met_rag', 'essential_total_rag', 'essential_percentage_rag', 'professional_met_rag', 'professional_total_rag', 'professional_percentage_rag', 'elite_met_rag', 'elite_total_rag', 'elite_percentage_rag', 'total_criteria_general', 'met_criteria_general', 'percentage_general', 'timestamp_general', 'essential_met_general', 'essential_total_general', 'essential_percentage_general', 'professional_met_general', 'professional_total_general', 'professional_percentage_general', 'elite_met_general', 'elite_total_general', 'elite_percentage_general', 'merge1', 'publication_external_id', 'publication_title', 'rag_score', 'rag_total', 'rag_percentage', 'general_score', 'general_total', 'general_percentage', 'combined_percentage', 'merge2', '_merge']

Found publication ID columns: ['publication_external_id']

=== ANALYZING publication_external_